---
# Production Code: Retrieval, Reflection & Generation Module

This section implements the **Retrieval**, **Self-Reflection**, and **Generation** parts of the team's workflow.

## Team Workflow Integration

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        FULL LANGGRAPH PIPELINE                          │
├─────────────────────────────────────────────────────────────────────────┤
│  [Chunking & Indexing]  →  Teammate's embeddings work                   │
│           ↓                                                             │
│  [Query Translation]    →  Pre-processing, compression                  │
│           ↓                                                             │
│  [Routing]              →  Route 0: Irrelevant → Template               │
│                            Route 1: Generic → Company JSON              │
│                            Route 2: Product → THIS MODULE               │
│           ↓                                                             │
│  [Query Construction]   →  Self-query, metadata filters                 │
│           ↓                                                             │
│  ┌─────────────────────────────────────────────────────────────────┐    │
│  │  THIS MODULE: Retrieval & Reflection                           │    │
│  │  • Hybrid retrieval (BM25 + Semantic) with metadata filtering  │    │
│  │  • Query Router (adaptive strategy selection)                  │    │
│  │  • Self-reflection: single query → if fails → RAG Fusion       │    │
│  │  • Light LLM for retrieval, Good LLM for reflection            │    │
│  └─────────────────────────────────────────────────────────────────┘    │
│           ↓                                                             │
│  [Generation]           →  Chat history + Persona (Teammate)            │
└─────────────────────────────────────────────────────────────────────────┘
```

## Input (from Query Construction node)
- `query`: The user's query (possibly translated/compressed)
- `metadata_filters`: Extracted metadata filters from self-query retrieval
- `route`: Should be "product" (Route 2) when reaching this module

## Output
- `response`: Generated answer from Good LLM
- `retrieved_docs`: List of relevant documents used
- `retrieval_status`: "success" | "no_relevant_docs" | "fusion_used"
- `generation_status`: "success" | "no_context" | "error"
- `sources`: Product sources cited in response


In [ ]:
## Dependencies

pip install langgraph rank-bm25 python-dotenv
pip install langchain-core langchain-community langchain-groq langchain-openai langchain-ollama
pip install langchain-huggingface langchain-chroma chromadb

%pip -q install "huggingface-hub>=0.33.4,<1.0.0"
%pip -q install "sentence-transformers>=2.2.0,<3.0.0"
%pip -q install "langchain-huggingface>=0.1.0"


In [ ]:
# =============================================================================
# RETRIEVAL & REFLECTION MODULE (Self-Contained for LangGraph Integration)
# =============================================================================

from __future__ import annotations
import os
import re
import time
import numpy as np
from pathlib import Path
from typing import Literal, Iterable, TypedDict, Optional, Any
from dataclasses import dataclass

from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.document_loaders import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langgraph.graph import StateGraph, START, END

load_dotenv()

# =============================================================================
# LLM FACTORY (Light + Good LLMs)
# =============================================================================
# Team Pattern: Factory design for LLMs via .env file
# - Light LLM: For retrieval operations (cheaper, faster)
# - Good LLM: For reflection/grading (more capable)
# =============================================================================

@dataclass
class LLMConfig:
    provider: str
    model: str


def _build_llm(cfg: LLMConfig):
    """
    Factory method to build LLM from config.
    Supports: Groq, OpenAI, Ollama, HuggingFace

    Team Note: Ollama recommended for local deployment with Llama/Qwen.
    """
    provider = (cfg.provider or "").lower().strip()

    if provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=cfg.model, temperature=0.0)

    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=cfg.model, temperature=0.0)

    if provider == "ollama":
        from langchain_ollama import ChatOllama
        base_url = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
        return ChatOllama(model=cfg.model, base_url=base_url, temperature=0.0)

    if provider in {"huggingface", "hf"}:
        from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
        endpoint = HuggingFaceEndpoint(
            repo_id=cfg.model,
            task="text-generation",
            max_new_tokens=1024,
            temperature=0.2,
        )
        return ChatHuggingFace(llm=endpoint)

    return None


# Initialize LLMs from .env
light_llm = _build_llm(LLMConfig(
    provider=os.environ.get("LIGHT_LLM_PROVIDER", ""),
    model=os.environ.get("LIGHT_LLM_MODEL", ""),
))
good_llm = _build_llm(LLMConfig(
    provider=os.environ.get("GOOD_LLM_PROVIDER", ""),
    model=os.environ.get("GOOD_LLM_MODEL", ""),
))

print(f"Light LLM: {getattr(light_llm, 'model_name', None) or getattr(light_llm, 'model', 'Not configured')}")
print(f"Good LLM: {getattr(good_llm, 'model_name', None) or getattr(good_llm, 'model', 'Not configured')}")

# =============================================================================
# DATA LOADING & INDEXING
# =============================================================================
# TODO [INTEGRATION]: Replace this section with teammate's Chunking & Indexing
#
# Teammate's work should provide:
# - langchain_docs: List of Document objects with enriched metadata
# - vector_store: Initialized vector store (ChromaDB recommended)
# - bm25: BM25Okapi index for lexical search
#
# Reference: Layout-Aware-Document-Extraction-Chunking-and-Indexing
# https://github.com/aws-samples/layout-aware-document-processing-and-retrieval-augmented-generation
# =============================================================================

def preprocess_for_bm25(text: str) -> list[str]:
    """BM25 tokenization: lowercase, remove punctuation."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    tokens = text.split()
    return [t for t in tokens if t and len(t) > 1]


def _passes_metadata_filter(metadata: dict, filters: dict | None) -> bool:
    """Check if document metadata passes the given filters."""
    if not filters:
        return True
    for key, value in filters.items():
        if value is None or value == "":
            continue
        if str(metadata.get(key, "")) != str(value):
            return False
    return True


def parse_metadata_from_content(content: str) -> dict:
    """Parse key: value pairs from CSVLoader's page_content format."""
    metadata = {}
    for line in content.split("\n"):
        if ": " in line:
            key, value = line.split(": ", 1)
            metadata[key.strip()] = value.strip()
    return metadata


# --- Load documents (REPLACE with teammate's chunking output) ---
CSV_PATH = Path(os.environ.get("CSV_PATH", "fashion.csv"))
print(f"\n📂 Loading data from: {CSV_PATH}")

loader = CSVLoader(file_path=str(CSV_PATH), encoding="utf-8")
langchain_docs = loader.load()

# Enrich metadata from content
for doc in langchain_docs:
    parsed = parse_metadata_from_content(doc.page_content)
    doc.metadata.update(parsed)

print(f"✅ Loaded {len(langchain_docs)} documents")

# --- Build BM25 index ---
print("\n🔧 Building BM25 index...")
corpus_tokens = [preprocess_for_bm25(doc.page_content) for doc in langchain_docs]
bm25 = BM25Okapi(corpus_tokens)
print(f"✅ BM25 index ready")

# --- Build Vector Store (ChromaDB) ---
# TODO [INTEGRATION]: Replace with teammate's embeddings
# Consider: ColBERT for improved embedding granularity
EMBED_MODEL_NAME = os.environ.get("EMBED_MODEL_NAME", "sentence-transformers/all-MiniLM-L6-v2")
print(f"\n🔧 Loading embeddings: {EMBED_MODEL_NAME}")
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL_NAME)

print("🔧 Building ChromaDB vector store...")
vector_store = Chroma.from_documents(
    documents=langchain_docs,
    embedding=embeddings,
    collection_name="fashion_products",
    persist_directory="./chroma_db"
)
print(f"✅ ChromaDB ready ({len(langchain_docs)} documents)")




In [ ]:
# =============================================================================
# SEARCH FUNCTIONS (Hybrid Retrieval)
# =============================================================================

def bm25_search(query: str, k: int = 10, filters: dict | None = None) -> list[dict]:
    """BM25 lexical search with metadata filtering."""
    tokens = preprocess_for_bm25(query)
    if not tokens:
        return []

    scores = bm25.get_scores(tokens)
    scored = []

    for idx, score in enumerate(scores):
        doc = langchain_docs[idx]
        if not _passes_metadata_filter(doc.metadata, filters):
            continue
        scored.append({
            "id": idx,
            "text": doc.page_content,
            "metadata": doc.metadata,
            "score": float(score),
            "source": "bm25",
        })

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:k]


def semantic_search(query: str, k: int = 10, filters: dict | None = None) -> list[dict]:
    """Semantic (vector) search using ChromaDB with metadata filtering."""
    search_k = k * 5 if filters else k
    results_with_scores = vector_store.similarity_search_with_score(query, k=search_k)

    results = []
    for doc, score in results_with_scores:
        metadata = doc.metadata
        if not _passes_metadata_filter(metadata, filters):
            continue
        results.append({
            "id": metadata.get("row", 0),
            "text": doc.page_content,
            "metadata": metadata,
            "score": float(score),
            "source": "semantic",
        })
        if len(results) >= k:
            break
    return results


def _rrf_fuse(result_lists: Iterable[list[dict]], k: int = 10, rrf_k: int = 60) -> list[dict]:
    """Reciprocal Rank Fusion to combine multiple result lists."""
    fused = {}
    for results in result_lists:
        for rank, item in enumerate(results, start=1):
            doc_id = item["id"]
            fused.setdefault(doc_id, {**item, "rrf_score": 0.0})
            fused[doc_id]["rrf_score"] += 1.0 / (rrf_k + rank)
    merged = list(fused.values())
    merged.sort(key=lambda x: x["rrf_score"], reverse=True)
    return merged[:k]


def hybrid_retrieve(query: str, k: int = 10, filters: dict | None = None) -> list[dict]:
    """Hybrid retrieval: BM25 + Semantic with RRF fusion."""
    bm25_results = bm25_search(query, k=k, filters=filters)
    semantic_results = semantic_search(query, k=k, filters=filters)
    return _rrf_fuse([bm25_results, semantic_results], k=k)

In [ ]:
# =============================================================================
# QUERY ROUTER (Adaptive Strategy Selection)
# =============================================================================

def classify_query_with_llm(query: str, llm=None) -> Literal["keyword", "semantic", "hybrid"]:
    """Use LLM to classify query into the best retrieval strategy."""
    USE_LLM_CLASSIFIER = True
    CLASSIFIER_LLM = light_llm
    if llm is None:
        return _classify_query_rules(query)

    prompt = f"""You are a query classifier for a product search system. Classify this query into ONE category:

KEYWORD: Use when query has specific product names, brands, colors, sizes, or structured attributes.
Examples: "Nike black shoes size 10", "women's red dress", "ADIDAS sandals for men"

SEMANTIC: Use when query describes intent, occasion, style, or asks for recommendations.
Examples: "something comfortable for summer", "outfit for a job interview"

HYBRID: Use when query mixes specific attributes WITH descriptive intent.
Examples: "elegant red dress for wedding", "comfortable Nike shoes for gym"

Query: "{query}"

Respond with exactly one word: keyword, semantic, or hybrid"""

    try:
        response = llm.invoke(prompt)
        result = response.content.strip().lower().split()[0]
        result = re.sub(r'[^a-z]', '', result)
        if result in ["keyword", "semantic", "hybrid"]:
            return result
        return _classify_query_rules(query)
    except Exception as e:
        print(f"⚠️ LLM classification failed: {str(e)[:50]}. Using rule-based fallback.")
        return _classify_query_rules(query)


def _classify_query_rules(query: str) -> Literal["keyword", "semantic", "hybrid"]:
    """Rule-based query classification (fallback)."""
    query_lower = query.lower().strip()
    words = query_lower.split()

    keyword_signals = 0
    semantic_signals = 0

    # Keyword signals
    if re.search(r'\b\d{4,}\b', query):  # Product IDs
        keyword_signals += 3
    if len(words) <= 3:
        keyword_signals += 1

    colors = ['red', 'blue', 'green', 'black', 'white', 'pink', 'yellow', 'purple', 'grey', 'brown']
    products = ['shirt', 'shoes', 'dress', 'pants', 'jeans', 'top', 'jacket', 'sandals', 'sneakers']
    brands = ['nike', 'adidas', 'puma', 'reebok', 'fila', 'clarks', 'gini and jony']

    has_color = any(c in query_lower for c in colors)
    has_product = any(p in query_lower for p in products)
    has_brand = any(b in query_lower for b in brands)

    if has_color and has_product:
        keyword_signals += 2
    if has_brand:
        keyword_signals += 2
    if any(query_lower.startswith(w) for w in ['do you have', 'is there', 'got any']):
        keyword_signals += 1

    # Semantic signals
    semantic_phrases = ['something for', 'recommend', 'suggest', 'goes with', 'comfy', 'occasion']
    for phrase in semantic_phrases:
        if phrase in query_lower:
            semantic_signals += 2
    if any(query_lower.startswith(w) for w in ['what should', 'what would', 'how do i style']):
        semantic_signals += 2
    if len(words) >= 6:
        semantic_signals += 1

    # Decision
    total = keyword_signals + semantic_signals
    if total == 0:
        return "keyword"
    keyword_ratio = keyword_signals / total

    if keyword_ratio >= 0.5:
        return "keyword"
    elif keyword_ratio <= 0.2:
        return "semantic"
    return "hybrid"


def classify_query(query: str, llm=None) -> Literal["keyword", "semantic", "hybrid"]:
    """Main classifier: Uses LLM if available, else rules."""
    active_llm = llm or CLASSIFIER_LLM
    if USE_LLM_CLASSIFIER and active_llm is not None:
        return classify_query_with_llm(query, active_llm)
    return _classify_query_rules(query)


def query_router(query: str, k: int = 10, filters: dict = None) -> list[dict]:
    """Adaptive retrieval: routes query to the best strategy."""
    query_type = classify_query(query)

    if query_type == "keyword":
        return bm25_search(query, k=k, filters=filters)
    elif query_type == "semantic":
        return semantic_search(query, k=k, filters=filters)
    else:
        return hybrid_retrieve(query, k=k, filters=filters)


# =============================================================================
# RAG FUSION (Multi-Query Expansion)
# =============================================================================

def generate_query_variations(query: str, llm=None, n: int = 3) -> list[str]:
    """Generate query variations for RAG Fusion using Light LLM."""
    _last_llm_call_time = 0
    LLM_CALL_DELAY = 1.0

    if llm is None:
        return [query]

    time_since_last = time.time() - _last_llm_call_time
    if time_since_last < LLM_CALL_DELAY:
        time.sleep(LLM_CALL_DELAY - time_since_last)

    try:
        prompt = f"Generate {n} diverse search queries that are paraphrases of: {query}\nReturn one per line."
        _last_llm_call_time = time.time()
        resp = llm.invoke([HumanMessage(content=prompt)])
        lines = [ln.strip() for ln in resp.content.splitlines() if ln.strip()]
        if query not in lines:
            lines.insert(0, query)
        return lines[:n]
    except Exception:
        return [query]


def rag_fusion_retrieve(query: str, k: int = 10, filters: dict = None) -> tuple[list[dict], list[str]]:
    """RAG Fusion: multiple query variations + RRF scoring."""
    variations = generate_query_variations(query, llm=light_llm, n=3)
    all_results = [query_router(q, k=k, filters=filters) for q in variations]
    fused = _rrf_fuse(all_results, k=k)
    return fused, variations


# =============================================================================
# REFLECTION (Relevance Check using Good LLM)
# =============================================================================

def check_relevance(query: str, docs: list[dict], llm=None) -> dict:
    """
    Check if retrieved documents are relevant using Good LLM.
    Uses rlm/rag-document-relevance prompt pattern.
    """
    if not docs:
        return {"is_relevant": False, "reason": "no_docs", "score": 0}
    if llm is None:
        return {"is_relevant": True, "reason": "no_llm_configured", "score": 1}

    # Prepare document snippets for grading
    documents = "\n\n".join(d["text"][:300] for d in docs[:3])

    system = """You are a grader assessing relevance of retrieved documents to a user question.
If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant.
Give a binary score 1 or 0, where 1 means relevant.
Respond in this format:
Score: <1 or 0>
Explanation: <your reasoning>"""

    user = f"""Retrieved documents: {documents}

User question: {query}

Score the documents for relevance."""

    try:
        resp = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)]).content

        # Parse response
        is_relevant = "score: 1" in resp.lower() or "score:1" in resp.lower()
        resp_lower = resp.lower()
        explanation = resp[resp_lower.find("explanation:") + len("explanation:"):].strip() if "explanation:" in resp_lower else resp

        return {
            "is_relevant": is_relevant,
            "reason": explanation,
            "score": 1 if is_relevant else 0
        }
    except Exception as e:
        return {"is_relevant": True, "reason": f"llm_error: {str(e)[:50]}", "score": 1}


In [ ]:
# =============================================================================
# COMPANY INFO & PERSONA (Placeholders for team integration)
# =============================================================================
# TODO [INTEGRATION]: Replace with actual company data from JSON/key-value store
# Team Workflow: Route 1 (Generic queries) uses company JSON directly
#                Route 2 (Product queries) uses this for persona in generation
# =============================================================================

# Company information - loaded from JSON or key-value store
# TODO [INTEGRATION]: Replace with actual company data loading
COMPANY_INFO = {
    "name": "Fashion Store",
    "description": "Your one-stop shop for trendy fashion items for the whole family.",
    "categories": ["Men's Wear", "Women's Wear", "Kids' Wear", "Footwear", "Accessories"],
    "tone": "friendly, helpful, fashion-forward",
    "policies": {
        "returns": "30-day return policy on unworn items",
        "shipping": "Free shipping on orders over $50",
    },
    # Schema for context formatting - DYNAMIC per company
    "context_schema": {
        "item_label": "Product",  # What to call each item (Product, Item, Service, etc.)
        "id_field": "ProductId",  # Unique identifier field
        "title_field": "ProductTitle",  # Main display field
        "display_fields": [
            # (field_name, display_label, optional)
            ("ProductTitle", "Title", False),
            ("ProductBrand", "Brand", True),
            ("Colour", "Color", True),
            ("Gender", "Gender", True),
            ("Category", "Category", True),
            ("SubCategory", "Subcategory", True),
            ("Price", "Price", True),
        ],
        "include_text": False,  # Whether to include raw text content
    }
}

# Company Persona - represents the company's brand identity
# TODO [INTEGRATION]: Load from company configuration (JSON/DB)
# This defines HOW the chatbot communicates based on brand identity
PERSONA = {
    # Brand Identity
    "brand_voice": "friendly, trendy, and approachable",
    "brand_values": ["quality", "affordability", "style for everyone"],
    
    # Target Audience
    "target_audience": {
        "demographics": "young adults and families",
        "interests": ["fashion", "trends", "value shopping"],
        "communication_preference": "casual, not overly formal",
    },
    
    # Communication Style (derived from brand + audience)
    "tone": "warm, helpful, and enthusiastic without being pushy",
    "language_style": "conversational, uses simple language, avoids jargon",
    
    # Response Guidelines
    "do": [
        "Be genuinely helpful and recommend products that fit their needs",
        "Use casual, friendly language that matches the brand vibe",
        "Mention specific product details (color, style, price when relevant)",
        "Acknowledge when you don't have what they're looking for",
        "Keep responses concise - respect their time",
    ],
    "dont": [
        "Be overly salesy or pushy",
        "Use formal/corporate language",
        "Make up information about products",
        "Ignore the customer's specific requirements",
    ],
}


def load_company_info(company_id: str = None) -> dict:
    """
    Load company information from storage.

    TODO [INTEGRATION]: Replace with actual company data loading
    - Could be from JSON files, database, or key-value store
    - company_id allows multi-tenant support
    """
    # Placeholder - return default company info
    return COMPANY_INFO


def load_persona(persona_id: str = None) -> dict:
    """
    Load persona configuration.

    TODO [INTEGRATION]: Replace with actual persona loading
    - Different personas for different use cases
    """
    return PERSONA


In [ ]:
# =============================================================================
# GENERATION NODE (Using Good LLM)
# =============================================================================
# Team Requirement: Generation with chat history and persona
# =============================================================================

GENERATION_PROMPT_TEMPLATE = """# Your Role
You are a customer service assistant for {company_name}.
{company_description}

# Brand Voice & Tone
{brand_voice}
{tone}

# Communication Guidelines
DO:
{do_guidelines}

DON'T:
{dont_guidelines}

# Target Audience
{target_audience}

# Instructions
Answer the user's question using ONLY the retrieved product information below.
- Understand what the user really needs
- Recommend the most relevant products from the context
- Write naturally, not as a list (unless they asked for options)
- Include specific details (name, color, price when relevant)
- If no relevant products found, acknowledge honestly and suggest alternatives

# Retrieved Products
<context>
{context}
</context>

# Chat History
{chat_history}

# User Question
{question}

# Your Response
"""

FALLBACK_PROMPT_TEMPLATE = """# Your Role
You are a customer service assistant for {company_name}.

# Brand Voice
{brand_voice}

# Situation
The user asked a question, but no relevant products were found in our catalog.

# Instructions
- Acknowledge you couldn't find exactly what they're looking for
- Suggest alternatives or ask clarifying questions
- Stay helpful and on-brand

# User Question
{question}

# Your Response
"""


def format_context(docs: list[dict], company_info: dict = None, max_docs: int = 5) -> str:
    """
    Dynamically format retrieved documents for the generation prompt.

    Uses company_info['context_schema'] to determine which fields to display.
    This allows the chatbot to work with any company's data schema.
    """
    if not docs:
        return "No items found."

    # Get schema from company info, or use sensible defaults
    if company_info and "context_schema" in company_info:
        schema = company_info["context_schema"]
    else:
        # Default: show all metadata fields dynamically
        schema = {
            "item_label": "Item",
            "title_field": None,
            "display_fields": None,  # None = show all fields
            "include_text": True,
        }

    item_label = schema.get("item_label", "Item")
    display_fields = schema.get("display_fields")
    include_text = schema.get("include_text", False)

    formatted = []
    for i, doc in enumerate(docs[:max_docs], 1):
        metadata = doc.get("metadata", {})
        text = doc.get("text", "")

        item_info = f"{item_label} {i}:\n"

        if display_fields:
            # Use configured fields
            for field_config in display_fields:
                if isinstance(field_config, tuple):
                    field_name, display_label, optional = field_config
                else:
                    field_name = field_config
                    display_label = field_config
                    optional = True

                value = metadata.get(field_name)
                if value or not optional:
                    item_info += f"  {display_label}: {value or 'N/A'}\n"
        else:
            # Dynamic: show all metadata fields
            for key, value in metadata.items():
                if value and key not in ['source', 'row']:  # Skip internal fields
                    # Convert field name to display label (CamelCase to spaces)
                    display_label = ''.join(' ' + c if c.isupper() else c for c in key).strip()
                    item_info += f"  {display_label}: {value}\n"

        if include_text and text:
            item_info += f"  Description: {text[:200]}...\n" if len(text) > 200 else f"  Description: {text}\n"

        formatted.append(item_info)

    return "\n".join(formatted)


def format_chat_history(history: list, max_turns: int = 5) -> str:
    """Format chat history for the prompt."""
    if not history:
        return "(No previous conversation)"

    formatted = []
    for msg in history[-max_turns*2:]:  # Last N turns (user + assistant pairs)
        role = msg.get("role", "user")
        content = msg.get("content", "")
        formatted.append(f"{role.capitalize()}: {content}")

    return "\n".join(formatted) if formatted else "(No previous conversation)"


def generate_response(
    query: str,
    docs: list[dict],
    chat_history: list = None,
    company_info: dict = None,
    persona: dict = None,
    llm = None
) -> dict:
    """
    Generate a response using the Good LLM.

    Args:
        query: User's question
        docs: Retrieved documents
        chat_history: Previous conversation
        company_info: Company details for context
        persona: Persona configuration
        llm: LLM to use (defaults to good_llm)

    Returns:
        dict with:
        - response: Generated text
        - sources: List of source documents used
        - status: "success" | "no_context" | "error"
    """
    llm = llm or good_llm
    company = company_info or load_company_info()
    persona_cfg = persona or load_persona()

    if llm is None:
        return {
            "response": "I'm sorry, I can't generate a response right now. Please try again later.",
            "sources": [],
            "status": "error_no_llm"
        }

    # Format inputs
    context = format_context(docs, company_info=company)
    history_str = format_chat_history(chat_history)
    
    # Format persona fields
    do_guidelines = "\n".join(f"- {g}" for g in persona_cfg.get("do", []))
    dont_guidelines = "\n".join(f"- {g}" for g in persona_cfg.get("dont", []))
    
    target_aud = persona_cfg.get("target_audience", {})
    target_audience_str = f"Demographics: {target_aud.get('demographics', 'general')}\nCommunication: {target_aud.get('communication_preference', 'friendly')}"

    # Choose prompt based on whether we have context
    if docs:
        prompt = GENERATION_PROMPT_TEMPLATE.format(
            company_name=company.get("name", "Our Store"),
            company_description=company.get("description", ""),
            brand_voice=f"Voice: {persona_cfg.get('brand_voice', 'helpful and friendly')}",
            tone=f"Tone: {persona_cfg.get('tone', 'warm and professional')}",
            do_guidelines=do_guidelines or "- Be helpful",
            dont_guidelines=dont_guidelines or "- Don't be pushy",
            target_audience=target_audience_str,
            context=context,
            chat_history=history_str,
            question=query
        )
    else:
        prompt = FALLBACK_PROMPT_TEMPLATE.format(
            company_name=company.get("name", "Our Store"),
            brand_voice=f"Voice: {persona_cfg.get('brand_voice', 'helpful and friendly')}",
            question=query
        )

    try:
        response = llm.invoke([HumanMessage(content=prompt)])

        # Extract source info from docs (using schema for field names)
        sources = []
        schema = company.get("context_schema", {})
        title_field = schema.get("title_field", "ProductTitle")
        id_field = schema.get("id_field", "ProductId")

        for doc in docs[:5]:
            metadata = doc.get("metadata", {})
            sources.append({
                "id": metadata.get(id_field) or metadata.get("id") or metadata.get("Id"),
                "title": metadata.get(title_field) or metadata.get("title") or metadata.get("name"),
            })

        return {
            "response": response.content.strip(),
            "sources": sources,
            "status": "success" if docs else "no_context"
        }
    except Exception as e:
        print(f"⚠️ Generation failed: {str(e)[:50]}")
        return {
            "response": "I apologize, but I encountered an issue generating a response. Please try again.",
            "sources": [],
            "status": f"error: {str(e)[:50]}"
        }




In [ ]:

# LANGGRAPH STATE & NODES
# =============================================================================
# Team Integration: This state flows through the full pipeline
# =============================================================================

class RetrievalState(TypedDict, total=False):
    """
    State for the Retrieval & Reflection module.

    INPUT FIELDS (from previous nodes):
    - query: User query (possibly translated/compressed from Query Translation)
    - metadata_filters: Extracted filters from Query Construction (self-query retrieval)
    - route: Should be "product" when reaching this module
    - chat_history: Previous messages (for context, passed to Generation)

    INTERNAL FIELDS (used within this module):
    - use_fusion: Whether to use RAG Fusion (set after first retrieval fails)
    - fusion_queries: Query variations generated for RAG Fusion

    OUTPUT FIELDS (for Generation node):
    - retrieved_docs: List of relevant documents
    - retrieval_status: "success" | "no_relevant_docs" | "fusion_used"
    - relevance_reason: Explanation from reflection LLM
    """
    # --- INPUT (from upstream nodes) ---
    query: str
    metadata_filters: Optional[dict]  # From Query Construction (self-query retrieval)
    route: Optional[str]              # Should be "product" for this module
    chat_history: Optional[list]      # Passed through to Generation

    # --- INTERNAL ---
    use_fusion: bool
    fusion_queries: list[str]

    # --- OUTPUT (for Generation node) ---
    retrieved_docs: list[dict]
    retrieval_status: str             # "success" | "no_relevant_docs" | "fusion_used"
    relevance_reason: str

    # --- GENERATION OUTPUT ---
    generated_response: Optional[str]
    generation_status: Optional[str]  # "success" | "no_context" | "error"
    sources: Optional[list[dict]]


def retrieve_node(state: RetrievalState) -> RetrievalState:
    """
    Retrieval node: Uses Query Router or RAG Fusion.
    Uses Light LLM for query classification.
    """
    query = state["query"]
    filters = state.get("metadata_filters")  # From Query Construction

    if state.get("use_fusion"):
        # RAG Fusion (retry after first pass failed)
        docs, variations = rag_fusion_retrieve(query, k=10, filters=filters)
        state["fusion_queries"] = variations
        state["retrieval_status"] = "fusion_used"
    else:
        # First pass: Single query with Query Router
        docs = query_router(query, k=10, filters=filters)
        state["retrieval_status"] = "success"

    state["retrieved_docs"] = docs
    return state


def reflect_node(state: RetrievalState) -> RetrievalState:
    """
    Reflection node: Check document relevance using Good LLM.
    """
    result = check_relevance(
        state["query"],
        state.get("retrieved_docs", []),
        llm=good_llm
    )
    state["is_relevant"] = result["is_relevant"]
    state["relevance_reason"] = result["reason"]
    return state


def enable_fusion_node(state: RetrievalState) -> RetrievalState:
    """Enable RAG Fusion for retry."""
    state["use_fusion"] = True
    return state


def finalize_node(state: RetrievalState) -> RetrievalState:
    """
    Finalize retrieval results for Generation node.

    OUTPUT for Generation:
    - retrieved_docs: Top 5 relevant documents
    - retrieval_status: "success" | "no_relevant_docs" | "fusion_used"
    - relevance_reason: Explanation
    """
    if state.get("is_relevant", False):
        # Keep top 5 docs for generation
        state["retrieved_docs"] = state.get("retrieved_docs", [])[:5]
        if state.get("use_fusion"):
            state["retrieval_status"] = "fusion_used"
        else:
            state["retrieval_status"] = "success"
    else:
        state["retrieval_status"] = "no_relevant_docs"
        state["retrieved_docs"] = []

    return state


def should_fuse(state: RetrievalState) -> str:
    """
    Decision: Retry with RAG Fusion if first pass not relevant.
    Team Requirement: Hybrid approach - single query first, then RAG Fusion.
    """
    if state.get("is_relevant", False):
        return "finalize"
    if not state.get("use_fusion"):
        return "fuse"  # Try RAG Fusion
    return "finalize"  # Already tried fusion, give up


# =============================================================================
# BUILD LANGGRAPH
# =============================================================================

graph = StateGraph(RetrievalState)

# Add nodes
graph.add_node("retrieve", retrieve_node)
graph.add_node("reflect", reflect_node)
graph.add_node("enable_fusion", enable_fusion_node)
graph.add_node("finalize", finalize_node)
graph.add_node("generate", generate_node)

# Add edges
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "reflect")
graph.add_conditional_edges("reflect", should_fuse, {
    "finalize": "finalize",
    "fuse": "enable_fusion",
})
graph.add_edge("enable_fusion", "retrieve")
graph.add_edge("finalize", "generate")
graph.add_edge("generate", END)

retrieval_graph = graph.compile()

print("\n✅ LangGraph Retrieval & Reflection module ready!")
print("")
print("Flow: retrieve → reflect → [if not relevant] → RAG fusion → reflect → finalize → generate")



In [ ]:

# =============================================================================
# PUBLIC API (for LangGraph Integration)
# =============================================================================
# This module supports TWO integration patterns:
#
# OPTION A: SUBGRAPH COMPOSITION
#   - Import `retrieval_subgraph` and add as a node in your main graph
#   - LangGraph handles state passing automatically
#   - Example: main_graph.add_node("retrieval", retrieval_subgraph)
#
# OPTION B: NODE FUNCTION
#   - Import `product_query_node` and add to your main graph
#   - Takes MainGraphState, returns updated state
#   - Example: main_graph.add_node("retrieval", product_query_node)
#
# =============================================================================

# -----------------------------------------------------------------------------
# OPTION A: Export subgraph for composition
# -----------------------------------------------------------------------------
retrieval_subgraph = retrieval_graph


# -----------------------------------------------------------------------------
# OPTION B: Node function for main graph integration
# -----------------------------------------------------------------------------
def product_query_node(state: dict) -> dict:
    """
    Node function for main LangGraph pipeline.

    Use this when you want to add retrieval+reflection+generation
    as a single node in your main graph.

    Expected state input (from upstream nodes):
        - query: str (from Query Translation)
        - metadata_filters: dict (from Query Construction)
        - chat_history: list (conversation context)
        - route: str (should be "product" for this node)

    State output (for downstream nodes / Messenger):
        - generated_response: str (the answer to send)
        - sources: list[dict] (cited products)
        - retrieved_docs: list[dict] (raw documents)
        - retrieval_status: str
        - generation_status: str
        - relevance_reason: str

    Usage in main graph:
        main_graph.add_node("product_retrieval", product_query_node)
        main_graph.add_edge("query_construction", "product_retrieval")
        main_graph.add_edge("product_retrieval", END)  # or next node
    """
    # Run the retrieval subgraph
    result = retrieval_graph.invoke({
        "query": state.get("query", ""),
        "metadata_filters": state.get("metadata_filters"),
        "route": state.get("route", "product"),
        "chat_history": state.get("chat_history", []),
    })

    # Return updated state (merge with existing state)
    return {
        **state,
        "generated_response": result.get("generated_response", ""),
        "sources": result.get("sources", []),
        "retrieved_docs": result.get("retrieved_docs", []),
        "retrieval_status": result.get("retrieval_status", "unknown"),
        "generation_status": result.get("generation_status", "unknown"),
        "relevance_reason": result.get("relevance_reason", ""),
    }





# =============================================================================
# EXPORTED COMPONENTS (for flexible integration)
# =============================================================================
# Individual node functions (if main graph wants fine-grained control):
#   - retrieve_node: Hybrid retrieval (BM25 + semantic)
#   - reflect_node: Relevance checking with self-reflection
#   - fuse_node: RAG fusion with multi-query expansion
#   - finalize_node: Document formatting
#   - generate_node: Response generation
#
# These are already defined above and can be imported directly.
# =============================================================================


In [ ]:
# -----------------------------------------------------------------------------
# Direct retrieval (no graph, no generation)
# -----------------------------------------------------------------------------
def retrieve_only(query: str, k: int = 10, filters: dict = None) -> list[dict]:
    """
    Direct retrieval without reflection/generation.
    Use this for testing or when you only need raw documents.
    """
    return query_router(query, k=k, filters=filters)

# =============================================================================
# QUICK TEST
# =============================================================================

print("\n" + "="*70)
print("🔍 QUICK TEST")
print("="*70)

test_query = "Do you have pink tops for girls?"
test_filters = {"Gender": "Girls", "Colour": "Pink"}  # From Query Construction

print(f"\nQuery: {test_query}")
print(f"Metadata Filters: {test_filters}")
print(f"Classification: {classify_query(test_query)}")

# Test direct retrieval
results = retrieve_only(test_query, k=3, filters=test_filters)
print(f"\n📦 Direct Retrieval: {len(results)} documents")
if results:
    print(f"  Top result: {results[0]['metadata'].get('ProductTitle', '')[:50]}...")

# Test full pipeline with generation
print("\n--- Full Pipeline (Retrieval + Reflection + Generation) ---")
result = retrieve_and_generate(test_query, metadata_filters=test_filters)
print(f"Retrieval Status: {result['retrieval_status']}")
print(f"Generation Status: {result['generation_status']}")
print(f"Documents Used: {len(result['retrieved_docs'])}")
print(f"\n📝 Generated Response:")
print("-" * 50)
print(result.get('response', 'No response generated'))
print("-" * 50)

if result.get('sources'):
    print("\n📚 Sources:")
    for src in result['sources'][:3]:
        print(f"  - {src.get('title', 'N/A')}")

print("\n" + "="*70)
print("✅ Full RAG Pipeline ready!")
print("   retrieve_and_generate(query, filters, chat_history) - Full pipeline")
print("   retrieve_only(query, filters) - Just retrieval")
print("   retrieve(query, k, filters) - Direct query router")
print("="*70)

---
# Evaluation Section

This section evaluates the retrieval strategies implemented above.

In [13]:
### Evaluation Metrics Utilities

import numpy as np
from typing import Callable

def calculate_retrieval_metrics(
    retrieved_docs: list[dict],
    ground_truth_ids: list[str],
    k: int = 10
) -> dict:
    """
    Calculate standard retrieval metrics.

    Args:
        retrieved_docs: List of retrieved documents with 'id' and 'metadata' fields
        ground_truth_ids: List of expected ProductIds (as strings)
        k: Number of top results to consider

    Returns:
        Dictionary of metrics
    """
    # Get retrieved ProductIds
    retrieved_ids = []
    for doc in retrieved_docs[:k]:
        # Try to get ProductId from metadata or id
        product_id = str(doc.get("metadata", {}).get("ProductId", doc.get("id", "")))
        retrieved_ids.append(product_id)

    # Convert ground truth to set for fast lookup
    gt_set = set(str(pid) for pid in ground_truth_ids)

    # Hit@k: Did we retrieve at least one relevant document?
    hits = [1 if rid in gt_set else 0 for rid in retrieved_ids]
    hit_at_k = 1.0 if any(hits) else 0.0

    # Recall@k: Proportion of ground truth docs retrieved
    retrieved_relevant = len(set(retrieved_ids) & gt_set)
    recall_at_k = retrieved_relevant / len(gt_set) if gt_set else 0.0

    # Precision@k: Proportion of retrieved docs that are relevant
    precision_at_k = retrieved_relevant / len(retrieved_ids) if retrieved_ids else 0.0

    # MRR: Reciprocal Rank of first relevant document
    mrr = 0.0
    for i, rid in enumerate(retrieved_ids, 1):
        if rid in gt_set:
            mrr = 1.0 / i
            break

    # MAP: Mean Average Precision
    relevant_count = 0
    precision_sum = 0.0
    for i, rid in enumerate(retrieved_ids, 1):
        if rid in gt_set:
            relevant_count += 1
            precision_sum += relevant_count / i
    map_score = precision_sum / len(gt_set) if gt_set else 0.0

    # NDCG@k: Normalized Discounted Cumulative Gain
    dcg = 0.0
    for i, rid in enumerate(retrieved_ids, 1):
        if rid in gt_set:
            dcg += 1.0 / np.log2(i + 1)

    # Ideal DCG (all relevant docs at top)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, min(len(gt_set), k) + 1))
    ndcg_at_k = dcg / idcg if idcg > 0 else 0.0

    return {
        "hit@k": hit_at_k,
        "recall@k": recall_at_k,
        "precision@k": precision_at_k,
        "mrr": mrr,
        "map": map_score,
        "ndcg@k": ndcg_at_k,
        "retrieved_ids": retrieved_ids,
        "ground_truth_ids": list(gt_set),
        "num_retrieved": len(retrieved_ids),
        "num_relevant_retrieved": retrieved_relevant,
    }


def evaluate_retrieval_strategy(
    retrieval_fn: Callable,
    eval_queries: list[dict],
    k: int = 5,
    strategy_name: str = "Strategy"
) -> pd.DataFrame:
    """
    Evaluate a retrieval strategy on all evaluation queries.

    Args:
        retrieval_fn: Function that takes (query, k, filters) and returns list of docs
        eval_queries: List of evaluation query dictionaries
        k: Number of results to retrieve
        strategy_name: Name of the strategy for display

    Returns:
        DataFrame with per-query and aggregate metrics
    """
    results = []

    print(f"\n{'='*60}")
    print(f"Evaluating: {strategy_name} (k={k})")
    print(f"{'='*60}")

    for i, eq in enumerate(eval_queries, 1):
        query = eq["query"]
        gt_ids = eq["ground_truth_product_ids"]
        filters = eq.get("filters", {})
        difficulty = eq.get("difficulty", "medium")
        query_type = eq.get("query_type", "specific")

        # Run retrieval
        try:
            docs = retrieval_fn(query, k=k, filters=filters if filters else None)
            metrics = calculate_retrieval_metrics(docs, gt_ids, k=k)
            metrics["query"] = query[:50] + "..." if len(query) > 50 else query
            metrics["strategy"] = strategy_name
            metrics["difficulty"] = difficulty
            metrics["query_type"] = query_type
            results.append(metrics)

            status = "✅" if metrics["hit@k"] > 0 else "❌"
            diff_tag = f"[{difficulty}]" if difficulty else ""
            print(f"{status} Query {i}/{len(eval_queries)} {diff_tag}: MRR={metrics['mrr']:.2f}, Recall={metrics['recall@k']:.2f}")
        except Exception as e:
            print(f"❌ Query {i} failed: {e}")
            results.append({
                "query": query[:50],
                "strategy": strategy_name,
                "difficulty": difficulty,
                "query_type": query_type,
                "hit@k": 0, "recall@k": 0, "precision@k": 0,
                "mrr": 0, "map": 0, "ndcg@k": 0, "error": str(e)
            })

    df = pd.DataFrame(results)
    return df


def print_evaluation_summary(df: pd.DataFrame, strategy_name: str = "Strategy"):
    """Print aggregate metrics summary"""
    print(f"\n{'='*60}")
    print(f"📊 {strategy_name} - EVALUATION SUMMARY")
    print(f"{'='*60}")
    print(f"\nTotal Queries: {len(df)}")
    print(f"\n🎯 Retrieval Metrics (averaged):")
    print(f"  • Hit@k:       {df['hit@k'].mean():.3f}  (found at least 1 relevant)")
    print(f"  • Recall@k:    {df['recall@k'].mean():.3f}  (completeness)")
    print(f"  • Precision@k: {df['precision@k'].mean():.3f}  (accuracy)")
    print(f"  • MRR:         {df['mrr'].mean():.3f}  (first relevant rank)")
    print(f"  • MAP:         {df['map'].mean():.3f}  (ranking quality)")
    print(f"  • NDCG@k:      {df['ndcg@k'].mean():.3f}  (graded relevance)")
    print(f"\n✅ Success Rate: {df['hit@k'].mean()*100:.1f}%")
    print(f"{'='*60}")

print("✅ Evaluation utilities loaded")

In [32]:
### Run Evaluation - Compare Retrieval Strategies

# ============================================================
# DEFINE ALL RETRIEVAL STRATEGIES
# ============================================================

# --- Basic Strategies ---
def bm25_only(query, k=5, filters=None):
    """BM25 lexical search only"""
    return bm25_search(query, k=k, filters=filters)

def semantic_only(query, k=5, filters=None):
    """Semantic search only"""
    return semantic_search(query, k=k, filters=filters)

def hybrid_rrf(query, k=5, filters=None):
    """Hybrid (BM25 + Semantic) with RRF fusion"""
    return hybrid_retrieve(query, k=k, filters=filters)

# --- Weighted Hybrid (adjustable weights) ---
def hybrid_weighted(query, k=5, filters=None, bm25_weight=0.3, semantic_weight=0.7):
    """Hybrid with weighted score combination instead of RRF"""
    bm25_results = bm25_search(query, k=k*2, filters=filters)
    semantic_results = semantic_search(query, k=k*2, filters=filters)

    # Normalize scores to 0-1 range
    if bm25_results:
        max_bm25 = max(r["score"] for r in bm25_results) or 1
        for r in bm25_results:
            r["norm_score"] = r["score"] / max_bm25

    if semantic_results:
        max_sem = max(r["score"] for r in semantic_results) or 1
        for r in semantic_results:
            r["norm_score"] = r["score"] / max_sem

    # Combine scores
    combined = {}
    for r in bm25_results:
        combined[r["id"]] = {**r, "final_score": r["norm_score"] * bm25_weight}

    for r in semantic_results:
        if r["id"] in combined:
            combined[r["id"]]["final_score"] += r["norm_score"] * semantic_weight
        else:
            combined[r["id"]] = {**r, "final_score": r["norm_score"] * semantic_weight}

    results = sorted(combined.values(), key=lambda x: x["final_score"], reverse=True)
    return results[:k]

def hybrid_bm25_heavy(query, k=5, filters=None):
    """Hybrid weighted 70% BM25 + 30% Semantic"""
    return hybrid_weighted(query, k=k, filters=filters, bm25_weight=0.7, semantic_weight=0.3)

def hybrid_semantic_heavy(query, k=5, filters=None):
    """Hybrid weighted 30% BM25 + 70% Semantic"""
    return hybrid_weighted(query, k=k, filters=filters, bm25_weight=0.3, semantic_weight=0.7)

def hybrid_balanced(query, k=5, filters=None):
    """Hybrid weighted 50% BM25 + 50% Semantic"""
    return hybrid_weighted(query, k=k, filters=filters, bm25_weight=0.5, semantic_weight=0.5)

# --- Re-ranking Strategies ---
def bm25_with_rerank(query, k=5, filters=None):
    """BM25 + Cross-encoder re-ranking"""
    docs = bm25_search(query, k=k*3, filters=filters)
    return rerank_cross_encoder(query, docs, enabled=True)[:k]

def semantic_with_rerank(query, k=5, filters=None):
    """Semantic search + Cross-encoder re-ranking"""
    docs = semantic_search(query, k=k*3, filters=filters)
    return rerank_cross_encoder(query, docs, enabled=True)[:k]

def hybrid_with_rerank(query, k=5, filters=None):
    """Hybrid + Cross-encoder re-ranking"""
    docs = hybrid_retrieve(query, k=k*3, filters=filters)
    return rerank_cross_encoder(query, docs, enabled=True)[:k]

# --- RAG Fusion ---
def rag_fusion_retrieval(query, k=5, filters=None):
    """RAG Fusion (multi-query + RRF)"""
    docs, _ = rag_fusion_retrieve(query, k=k, filters=filters, llm=light_llm)
    return docs

# --- Larger K retrieval ---
def hybrid_k10(query, k=5, filters=None):
    """Hybrid with larger initial retrieval (k=10), return top 5"""
    docs = hybrid_retrieve(query, k=10, filters=filters)
    return docs[:k]

# ============================================================
# SELECT STRATEGIES TO EVALUATE
# ============================================================

# Choose which strategies to test (comment/uncomment as needed)
STRATEGIES_TO_TEST = {
    # Basic (always test)
    "BM25 Only": bm25_only,
    "Semantic Only": semantic_only,
    "Hybrid (RRF)": hybrid_rrf,

    # Weighted combinations
    "Hybrid (BM25 70%)": hybrid_bm25_heavy,
    "Hybrid (Balanced 50/50)": hybrid_balanced,
    "Hybrid (Semantic 70%)": hybrid_semantic_heavy,

    # Re-ranking (requires CrossEncoder)
    "BM25 + Rerank": bm25_with_rerank,
    "Semantic + Rerank": semantic_with_rerank,
    "Hybrid + Rerank": hybrid_with_rerank,

    # Query Router (Adaptive Retrieval) - NEW!
    "Query Router": query_router,
    "Query Router + Rerank": query_router_with_rerank,

    # Multi-query (disabled - requires LLM credits)
    # "RAG Fusion": rag_fusion_retrieval,
}

# Evaluation settings
EVAL_K = 10  # Number of results to retrieve

# ============================================================
# RUN EVALUATION
# ============================================================

all_results = []

for strategy_name, strategy_fn in STRATEGIES_TO_TEST.items():
    df = evaluate_retrieval_strategy(strategy_fn, eval_queries, k=EVAL_K, strategy_name=strategy_name)
    all_results.append(df)

print(f"\n✅ Evaluation complete for {len(STRATEGIES_TO_TEST)} strategies!")

### Test with Realistic Messenger-Style Queries

Simulate how real users would query on Messenger vs the current structured evaluation.

In [25]:
### Compare BM25 vs Query Router on Realistic Messenger Queries

# Realistic queries that users would actually type on Messenger
messenger_style_queries = [
    # Semantic - descriptive, intent-based
    {"query": "I need something for a summer party", "type": "semantic"},
    {"query": "looking for comfortable everyday wear", "type": "semantic"},
    {"query": "what do you recommend for a beach vacation?", "type": "semantic"},
    {"query": "something elegant for a dinner date", "type": "semantic"},
    {"query": "casual outfit for working from home", "type": "semantic"},
    {"query": "sporty clothes for the gym", "type": "semantic"},

    # Keyword - specific product searches
    {"query": "Nike shoes size 10", "type": "keyword"},
    {"query": "black formal shirt", "type": "keyword"},
    {"query": "girls pink dress", "type": "keyword"},
    {"query": "men blue jeans", "type": "keyword"},

    # Hybrid - mix of specific + descriptive
    {"query": "red dress for wedding guest", "type": "hybrid"},
    {"query": "comfortable white sneakers for walking", "type": "hybrid"},
    {"query": "warm jacket for winter travel", "type": "hybrid"},
    {"query": "cute top for teenage girl", "type": "hybrid"},
]

print("🔍 REALISTIC MESSENGER QUERY COMPARISON")
print("=" * 90)
print(f"Testing {len(messenger_style_queries)} production-style queries\n")

results_comparison = []

for mq in messenger_style_queries:
    query = mq["query"]
    expected_type = mq["type"]

    # Classify and route
    classified_as = classify_query(query)

    # Get results from both strategies
    bm25_results = bm25_search(query, k=5)
    router_results = query_router(query, k=5)

    # Check if routing matches expected
    route_correct = classified_as == expected_type

    results_comparison.append({
        "query": query,
        "expected": expected_type,
        "routed_to": classified_as,
        "correct": route_correct,
        "bm25_top": bm25_results[0]["text"][:60] + "..." if bm25_results else "No results",
        "router_top": router_results[0]["text"][:60] + "..." if router_results else "No results",
    })

# Summary
correct_routes = sum(1 for r in results_comparison if r["correct"])
print(f"📊 Routing Accuracy: {correct_routes}/{len(results_comparison)} ({correct_routes/len(results_comparison)*100:.0f}%)")
print()

# Show each query
print(f"{'Query':<45} {'Expected':<10} {'Routed':<10} {'Match'}")
print("-" * 90)
for r in results_comparison:
    match_icon = "✅" if r["correct"] else "❌"
    print(f"{r['query'][:44]:<45} {r['expected']:<10} {r['routed_to']:<10} {match_icon}")

print("\n" + "=" * 90)
print("📋 SAMPLE RESULTS COMPARISON (first 5 queries):")
print("-" * 90)

for r in results_comparison[:5]:
    print(f"\n🔍 Query: \"{r['query']}\"")
    print(f"   Routed to: {r['routed_to']} (expected: {r['expected']})")
    print(f"   BM25 top:   {r['bm25_top']}")
    print(f"   Router top: {r['router_top']}")

print("\n" + "=" * 90)
print("💡 INSIGHT:")
print("   In production, Query Router helps because:")
print("   • Semantic queries get better conceptual matches")
print("   • Keyword queries still get exact BM25 matches")
print("   • The router adapts to user intent automatically")
print("=" * 90)

In [26]:
### Comparison Table - All Strategies

# Combine all results
df_all = pd.concat(all_results, ignore_index=True)

# Create comparison summary
comparison = df_all.groupby("strategy").agg({
    "hit@k": "mean",
    "recall@k": "mean",
    "precision@k": "mean",
    "mrr": "mean",
    "map": "mean",
    "ndcg@k": "mean",
}).round(3)

# Sort by MRR (or your preferred metric)
comparison = comparison.sort_values("mrr", ascending=False)

print("\n" + "=" * 80)
print("📊 RETRIEVAL STRATEGY COMPARISON")
print("=" * 80)
print(f"\nEvaluation: {len(eval_queries)} queries, k={EVAL_K}")
print(f"Strategies tested: {len(comparison)}")
print("\n")

# Display comparison table
display(comparison)

# Find best strategy for each metric
print("\n🏆 Best Strategy per Metric:")
for col in ["hit@k", "recall@k", "precision@k", "mrr", "map", "ndcg@k"]:
    best = comparison[col].idxmax()
    best_val = comparison.loc[best, col]
    print(f"  • {col:12s}: {best} ({best_val:.3f})")

# Overall winner
best_mrr = comparison["mrr"].idxmax()
best_recall = comparison["recall@k"].idxmax()
print(f"\n🥇 RECOMMENDED: {best_mrr} (highest MRR: {comparison.loc[best_mrr, 'mrr']:.3f})")

# Category analysis
print("\n" + "=" * 80)
print("📈 STRATEGY CATEGORY ANALYSIS")
print("=" * 80)

# Group by strategy type
bm25_strategies = [s for s in comparison.index if "BM25" in s and "Hybrid" not in s]
semantic_strategies = [s for s in comparison.index if "Semantic" in s and "Hybrid" not in s]
hybrid_strategies = [s for s in comparison.index if "Hybrid" in s]
fusion_strategies = [s for s in comparison.index if "Fusion" in s]

categories = [
    ("BM25-based", bm25_strategies),
    ("Semantic-based", semantic_strategies),
    ("Hybrid", hybrid_strategies),
    ("Multi-query (Fusion)", fusion_strategies),
]

for cat_name, strategies in categories:
    if strategies:
        cat_df = comparison.loc[strategies]
        avg_mrr = cat_df["mrr"].mean()
        avg_recall = cat_df["recall@k"].mean()
        best_in_cat = cat_df["mrr"].idxmax()
        print(f"\n{cat_name}:")
        print(f"  Avg MRR: {avg_mrr:.3f} | Avg Recall: {avg_recall:.3f}")
        print(f"  Best: {best_in_cat} (MRR: {cat_df.loc[best_in_cat, 'mrr']:.3f})")

# Calculate improvement from baseline (BM25)
print("\n📈 Improvement over BM25 Baseline:")
if "BM25 Only" in comparison.index:
    baseline = comparison.loc["BM25 Only"]
    for strategy in comparison.index:
        if strategy != "BM25 Only":
            mrr_diff = comparison.loc[strategy, "mrr"] - baseline["mrr"]
            recall_diff = comparison.loc[strategy, "recall@k"] - baseline["recall@k"]
            print(f"  • {strategy}: MRR {mrr_diff:+.3f}, Recall {recall_diff:+.3f}")

# Breakdown by difficulty level
if "difficulty" in df_all.columns:
    print("\n📊 Performance by Difficulty Level:")
    for difficulty in ["easy", "medium", "hard"]:
        diff_df = df_all[df_all["difficulty"] == difficulty]
        if len(diff_df) > 0:
            diff_comparison = diff_df.groupby("strategy").agg({"hit@k": "mean", "mrr": "mean"}).round(3)
            print(f"\n  [{difficulty.upper()}] ({len(diff_df)//len(comparison)} queries)")
            for strategy in diff_comparison.index:
                hit = diff_comparison.loc[strategy, "hit@k"]
                mrr = diff_comparison.loc[strategy, "mrr"]
                print(f"    • {strategy}: Hit={hit:.1%}, MRR={mrr:.3f}")

In [27]:
### Visualization - Strategy Comparison

import matplotlib.pyplot as plt
import numpy as np

# Create comparison bar chart
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Metrics to plot
metrics_to_plot = [
    ("MRR (Mean Reciprocal Rank)", "mrr"),
    ("Recall@k", "recall@k"),
    ("Hit@k", "hit@k"),
    ("NDCG@k", "ndcg@k"),
]

strategies = comparison.index.tolist()
colors = plt.cm.Set2(np.linspace(0, 1, len(strategies)))

for ax, (title, metric) in zip(axes.flatten(), metrics_to_plot):
    values = comparison[metric].values
    bars = ax.bar(range(len(strategies)), values, color=colors)
    ax.set_xlabel("Strategy", fontsize=10)
    ax.set_ylabel(title, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(strategies)))
    ax.set_xticklabels(strategies, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 1)
    ax.axhline(y=values.max(), color='green', linestyle='--', alpha=0.3)

    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.suptitle('Retrieval Strategy Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.show()

# Radar chart for comprehensive view (if more than 2 strategies)
if len(strategies) >= 3:
    from math import pi

    # Select key metrics
    radar_metrics = ["mrr", "recall@k", "hit@k", "ndcg@k", "precision@k", "map"]

    # Number of variables
    N = len(radar_metrics)

    # Angle for each metric
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]  # complete the circle

    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

    for i, strategy in enumerate(strategies[:6]):  # Max 6 strategies for readability
        values = comparison.loc[strategy, radar_metrics].values.tolist()
        values += values[:1]  # complete the circle
        ax.plot(angles, values, 'o-', linewidth=2, label=strategy, color=colors[i])
        ax.fill(angles, values, alpha=0.1, color=colors[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_metrics, fontsize=10)
    ax.set_ylim(0, 1)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1))
    plt.title('Strategy Performance Radar', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("\n✅ Visualization complete!")

### Optimal k Analysis

Evaluate different k values to find the optimal retrieval depth for your dataset.

In [28]:
### Find Optimal k Value

# Test different k values to find the optimal retrieval depth
K_VALUES_TO_TEST = [3, 5, 7, 10, 15, 20]

# Pick the best performing strategy from previous evaluation for k analysis
# Note: 'strategy' is the index of comparison DataFrame (from groupby)
best_strategy_name = comparison.index[0]
best_strategy_fn = query_router

print(f"📊 Analyzing optimal k for: {best_strategy_name}")
print(f"   Testing k values: {K_VALUES_TO_TEST}")
print("=" * 60)

k_results = []

for k in K_VALUES_TO_TEST:
    df = evaluate_retrieval_strategy(best_strategy_fn, eval_queries, k=k, strategy_name=f"k={k}")

    metrics = {
        'k': k,
        'hit@k': df['hit@k'].mean(),
        'recall@k': df['recall@k'].mean(),
        'precision@k': df['precision@k'].mean(),
        'mrr': df['mrr'].mean(),
        'map': df['map'].mean(),
        'ndcg@k': df['ndcg@k'].mean()
    }
    k_results.append(metrics)
    print(f"  k={k:2d} | Hit: {metrics['hit@k']:.3f} | Recall: {metrics['recall@k']:.3f} | MRR: {metrics['mrr']:.3f} | NDCG: {metrics['ndcg@k']:.3f}")

df_k_analysis = pd.DataFrame(k_results)

print("\n" + "=" * 60)
print("📈 K-VALUE ANALYSIS RESULTS")
print("=" * 60)
display(df_k_analysis.round(3))

# Find optimal k based on different criteria
optimal_hit = df_k_analysis.loc[df_k_analysis['hit@k'].idxmax(), 'k']
optimal_recall = df_k_analysis.loc[df_k_analysis['recall@k'].idxmax(), 'k']
optimal_mrr = df_k_analysis.loc[df_k_analysis['mrr'].idxmax(), 'k']
optimal_ndcg = df_k_analysis.loc[df_k_analysis['ndcg@k'].idxmax(), 'k']

print(f"\n🎯 Optimal k by metric:")
print(f"   • Best Hit@k:    k={int(optimal_hit)}")
print(f"   • Best Recall@k: k={int(optimal_recall)}")
print(f"   • Best MRR:      k={int(optimal_mrr)}")
print(f"   • Best NDCG@k:   k={int(optimal_ndcg)}")

# Calculate efficiency score (recall vs k trade-off)
df_k_analysis['efficiency'] = df_k_analysis['recall@k'] / df_k_analysis['k']
optimal_efficiency = df_k_analysis.loc[df_k_analysis['efficiency'].idxmax(), 'k']
print(f"   • Best Efficiency (recall/k): k={int(optimal_efficiency)}")

# Find elbow point (diminishing returns)
recall_gains = df_k_analysis['recall@k'].diff().fillna(0)
df_k_analysis['recall_gain'] = recall_gains
elbow_idx = recall_gains[recall_gains < 0.02].first_valid_index()
if elbow_idx is not None and elbow_idx > 0:
    elbow_k = df_k_analysis.loc[elbow_idx - 1, 'k']
    print(f"   • Elbow point (diminishing returns): k={int(elbow_k)}")

print(f"\n💡 Recommendation:")
if optimal_mrr == optimal_ndcg:
    print(f"   Use k={int(optimal_mrr)} for best ranking quality (MRR & NDCG agree)")
else:
    print(f"   Use k={int(optimal_ndcg)} for best overall ranking (NDCG)")
    print(f"   Use k={int(optimal_efficiency)} for best efficiency (recall per document)")

In [29]:
### Visualize k Analysis

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Main metrics vs k
ax1 = axes[0]
ax1.plot(df_k_analysis['k'], df_k_analysis['hit@k'], 'o-', label='Hit@k', linewidth=2, markersize=8)
ax1.plot(df_k_analysis['k'], df_k_analysis['recall@k'], 's-', label='Recall@k', linewidth=2, markersize=8)
ax1.plot(df_k_analysis['k'], df_k_analysis['precision@k'], '^-', label='Precision@k', linewidth=2, markersize=8)
ax1.set_xlabel('k (number of retrieved documents)', fontsize=11)
ax1.set_ylabel('Score', fontsize=11)
ax1.set_title('Hit, Recall, Precision vs k', fontsize=12, fontweight='bold')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)
ax1.set_xticks(df_k_analysis['k'])

# Plot 2: Ranking metrics vs k
ax2 = axes[1]
ax2.plot(df_k_analysis['k'], df_k_analysis['mrr'], 'o-', label='MRR', linewidth=2, markersize=8, color='green')
ax2.plot(df_k_analysis['k'], df_k_analysis['ndcg@k'], 's-', label='NDCG@k', linewidth=2, markersize=8, color='purple')
ax2.plot(df_k_analysis['k'], df_k_analysis['map'], '^-', label='MAP', linewidth=2, markersize=8, color='orange')
ax2.set_xlabel('k (number of retrieved documents)', fontsize=11)
ax2.set_ylabel('Score', fontsize=11)
ax2.set_title('Ranking Metrics vs k', fontsize=12, fontweight='bold')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(df_k_analysis['k'])

# Plot 3: Recall gain (to identify elbow)
ax3 = axes[2]
bars = ax3.bar(df_k_analysis['k'].astype(str), df_k_analysis['recall_gain'], color='steelblue', edgecolor='navy')
ax3.axhline(y=0.02, color='red', linestyle='--', label='Diminishing returns threshold (0.02)')
ax3.set_xlabel('k (number of retrieved documents)', fontsize=11)
ax3.set_ylabel('Recall Gain from previous k', fontsize=11)
ax3.set_title('Marginal Recall Gain (Elbow Detection)', fontsize=12, fontweight='bold')
ax3.legend(loc='best')
ax3.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.suptitle(f'Optimal k Analysis for {best_strategy_name}', fontsize=14, fontweight='bold', y=1.02)
plt.show()

print(f"\n📊 Interpretation:")
print(f"   • Left plot: Higher k increases recall but decreases precision")
print(f"   • Middle plot: Ranking quality (MRR/NDCG) shows optimal k for your use case")
print(f"   • Right plot: When recall gain drops below threshold, increasing k has diminishing returns")

#### Final Performance Summary

This summary provides metrics in the same format as the baseline evaluation for fair comparison.

In [37]:
### Final Performance Summary (for comparison with Baseline)

# Use Query Router as the selected strategy for production
best_strategy = "Query Router"
best_metrics = comparison.loc[best_strategy]

print("\n" + "=" * 80)
print("📊 RETRIEVAL EVALUATION COMPLETE")
print("=" * 80)
print(f"\nConfiguration:")
print(f"  • Best Strategy: {best_strategy}")
print(f"  • Embeddings: sentence-transformers/all-MiniLM-L6-v2")
print(f"  • k={EVAL_K}")
print(f"  • Total Queries: {len(eval_queries)}")
print(f"  • Strategies Tested: {len(comparison)}")
print(f"\n🎯 Best Strategy Metrics ({best_strategy}):")
print(f"  • Hit@k:       {best_metrics['hit@k']:.3f}")
print(f"  • Recall@k:    {best_metrics['recall@k']:.3f}")
print(f"  • Precision@k: {best_metrics['precision@k']:.3f}")
print(f"  • MRR:         {best_metrics['mrr']:.3f}")
print(f"  • MAP:         {best_metrics['map']:.3f}")
print(f"  • NDCG@k:      {best_metrics['ndcg@k']:.3f}")
print(f"\n✅ Success Rate: {best_metrics['hit@k']*100:.1f}%")

# Performance by difficulty for best strategy
if "difficulty" in df_all.columns:
    print(f"\n📊 {best_strategy} Performance by Difficulty:")
    best_df = df_all[df_all["strategy"] == best_strategy]
    for difficulty in ["easy", "medium", "hard"]:
        diff_df = best_df[best_df["difficulty"] == difficulty]
        if len(diff_df) > 0:
            print(f"\n  [{difficulty.upper()}] ({len(diff_df)} queries)")
            print(f"    • Hit@k:    {diff_df['hit@k'].mean():.3f}")
            print(f"    • Recall:   {diff_df['recall@k'].mean():.3f}")
            print(f"    • MRR:      {diff_df['mrr'].mean():.3f}")

print("=" * 80)

In [33]:
### Detailed Per-Query Results

def show_detailed_results(df: pd.DataFrame, strategy_name: str):
    """Show detailed results for a specific strategy"""
    strategy_df = df[df["strategy"] == strategy_name].copy()

    print(f"\n{'='*70}")
    print(f"📋 Detailed Results: {strategy_name}")
    print(f"{'='*70}")

    for i, row in strategy_df.iterrows():
        status = "✅" if row["hit@k"] > 0 else "❌"
        difficulty = row.get('difficulty', 'N/A')
        query_type = row.get('query_type', 'N/A')
        print(f"\n{status} [{difficulty}] Query: {row['query']}")
        print(f"   Type: {query_type}")
        print(f"   Expected: {row.get('ground_truth_ids', 'N/A')}")
        print(f"   Retrieved: {row.get('retrieved_ids', 'N/A')[:5]}")
        print(f"   Metrics: MRR={row['mrr']:.2f}, Recall={row['recall@k']:.2f}, Precision={row['precision@k']:.2f}")

# Show detailed results for best and baseline strategies
best_strategy = comparison.index[1]  # Best by MRR
print_evaluation_summary(df_all[df_all["strategy"] == best_strategy], best_strategy)
show_detailed_results(df_all, best_strategy)

# Also show BM25 baseline for comparison
if "BM25 Only" in comparison.index and best_strategy != "BM25 Only":
    print("\n\n" + "="*70)
    print("📋 Baseline Comparison (BM25 Only)")
    print("="*70)
    show_detailed_results(df_all, "BM25 Only")